# Predicting Fast-Growing Firms (2025)
Data Analysis 3 - Assignment 2  
Submitted by: Balint Decsi (ID: ), Zariza Chowdhury (ID: 2500086)    
Deadline: 15 February 2026

## Objective
The objective of this assignment is to build and evaluate predictive models for identifying fast-growing firms using the Bisnode firms panel data.      
The analysis focuses on probability prediction, classification under an explicit loss function, and comparison of model performance across industries.

**Reproducibility and Project Structure**

This notebook is designed to be run from within the `Assignment-2/` directory.  
All file paths are specified as relative paths to ensure reproducibility across environments.  
Raw data, cleaned data, outputs, and reports follow a structured folder layout consistent with prior assignments.

## Modeling Overview and Target Choice

The fast-growth target used throughout this notebook is based on a one-year growth window (2013 vs 2012), defined as firms in the top decile of log sales growth over this period.      

This choice balances interpretability and sample size, while aligning with standard approaches in the corporate finance literature that emphasize short-horizon growth dynamics and mean reversion.

This notebook focuses on the modeling and evaluation stages of the analysis.        

## Task 1

Task 1 estimates and compares multiple predictive models for fast growth, first through probability prediction and then through classification under an explicit loss function.         

At least one parametric (logit) and one non-parametric (random forest) model are evaluated, with model selection based on cross-validated out-of-sample performance.

### Part I: Probability Prediction

This part estimates predictive models for the probability that a firm is fast-growing.      
Model performance is evaluated using cross-validation, and the preferred model is selected based on out-of-sample performance.

##### Step 1: Environment setup and imports

In [49]:
# Import necessary libraries and set up the environment
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import time

# Modeling + evaluation
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    make_scorer,
    average_precision_score,
    log_loss,
    brier_score_loss,
    confusion_matrix,
)

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

In [41]:
# Define the paths (run from Assignment-2/)
base_dir = os.getcwd()

data_clean = os.path.join(base_dir, "Data", "Cleaned")
output_plots = os.path.join(base_dir, "Output", "Plots")
output_tables = os.path.join(base_dir, "Output", "Tables")

for p in [output_plots, output_tables]:
    os.makedirs(p, exist_ok=True)

data_clean, output_plots, output_tables

('/Users/zarizachowdhury/Documents/Data-Analysis-3/Assignment-2/Data/Cleaned',
 '/Users/zarizachowdhury/Documents/Data-Analysis-3/Assignment-2/Output/Plots',
 '/Users/zarizachowdhury/Documents/Data-Analysis-3/Assignment-2/Output/Tables')

In [42]:
# Helpers for clean overwrites (tables & figures)
def overwrite_csv(df, filepath, index=False):
    if os.path.exists(filepath):
        os.remove(filepath)
    df.to_csv(filepath, index=index)

def overwrite_fig(filepath):
    if os.path.exists(filepath):
        os.remove(filepath)

##### Step 2: Load the final modeling dataset

In [43]:
# Load final modeling dataset
model_path = os.path.join(data_clean, "bisnode_firms_clean.csv")
df = pd.read_csv(model_path)

print("Loaded dataset from:", model_path)
print("Dataset shape:", df.shape)

# Quick sanity check
df["fast_growth"].value_counts(normalize=True)

Loaded dataset from: /Users/zarizachowdhury/Documents/Data-Analysis-3/Assignment-2/Data/Cleaned/bisnode_firms_clean.csv
Dataset shape: (19135, 123)


fast_growth
0    0.899974
1    0.100026
Name: proportion, dtype: float64

##### Step 3: Define the target and feature set

Separate the target variable (`fast_growth`) from the predictors and remove identifiers and outcome-year fields to avoid leakage.

In [ ]:
# Define X and y
y = df["fast_growth"].astype(int)

# Drop identifiers + target + outcome-year variables (avoid leakage)
drop_cols = [
    "comp_id", "year",
    "fast_growth",
    "growth_log_2013_2012",
    "sales_2013", "sales_mil_2013", "sales_mil_log_2013",
]

X = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

# Identify categorical vs numeric columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# Drop columns that are entirely missing (causes imputer warnings)
all_missing_cols = X.columns[X.isna().all()].tolist()

print("All-missing columns dropped:", all_missing_cols)

X = X.drop(columns=all_missing_cols)

# Recompute column lists after dropping
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("X shape:", X.shape)
print("Number of numeric features:", len(num_cols))
print("Number of categorical features:", len(cat_cols))
print("Fast-growth rate:", y.mean())

cat_cols[:10]

All-missing columns dropped: ['D']
X shape: (19135, 115)
Number of numeric features: 106
Number of categorical features: 9
Fast-growth rate: 0.10002613012803763


['begin',
 'end',
 'gender',
 'origin',
 'region_m',
 'founded_date',
 'exit_date',
 'gender_m',
 'm_region_loc']

##### Step 4: Build the preprocessing pipeline

Create a reusable preprocessing pipeline:       
- numeric variables are imputed with the mean      
- categorical variables are imputed with the most frequent value and one-hot encoded

In [45]:
# Preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop"
)

preprocess

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer', SimpleImputer())]),
                                 ['amort', 'curr_assets', 'curr_liab',
                                  'extra_exp', 'extra_inc', 'extra_profit_loss',
                                  'fixed_assets', 'inc_bef_tax',
                                  'intang_assets', 'inventories', 'liq_assets',
                                  'material_exp', 'personnel_exp',
                                  'profit_loss_year', 'sales', 'share_eq',
                                  'subscribed_cap', 'tang_assets',
                                  'balsheet_flag...
                                  'balsheet_notfullyear', 'founded_year',
                                  'exit_year', 'ceo_count', 'foreign', 'female',
                                  'birth_year', 'inoffice_days', 'nace_main',
                                  'ind2', ...]),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['begin', 'end', 'gender', 'origin',
                                  'region_m', 'founded_date', 'exit_date',
                                  'gender_m', 'm_region_loc'])])

##### Choosing Candidate Models

To evaluate different approaches to probability prediction, I estimate three complementary models.  

- A standard logit with L2 regularization provides a transparent parametric baseline.
- A logit with L1 (LASSO) regularization allows for automatic feature selection in a high-dimensional setting.
- Finally, a random forest captures non-linearities and interactions without requiring explicit specification, serving as a flexible non-parametric benchmark.

##### Step 5: Baseline Probability Model (Logit) with Cross-Validation

Start with a transparent baseline model to predict fast-growth probabilities.       
A logit provides interpretability and a benchmark for more flexible models.

In [50]:
# Logit (L2) baseline + CV

# Start timer
start_time = time.time()

# 5-fold stratified CV (fixed seed for reproducibility)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic regression with L2 regularization
logit_l2 = LogisticRegression(
    penalty="l2",
    solver="saga",
    max_iter=5000,
    n_jobs=-1
)

# Pipeline: preprocessing + model
logit_l2_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", logit_l2),
])

# Cross-validated predicted probabilities
logit_l2_proba = cross_val_predict(
    logit_l2_pipe,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Runtime
runtime_l2 = time.time() - start_time

# Performance metrics
logit_l2_metrics = pd.DataFrame([{
    "model": "Logit (L2)",
    "roc_auc": roc_auc_score(y, logit_l2_proba),
    "avg_precision": average_precision_score(y, logit_l2_proba),
    "log_loss": log_loss(y, logit_l2_proba),
    "brier": brier_score_loss(y, logit_l2_proba),
    "runtime_sec": runtime_l2
}])

logit_l2_metrics

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,model,roc_auc,avg_precision,log_loss,brier,runtime_sec
0,Logit (L2),0.752954,0.268562,0.432966,0.120592,50.682915


##### Step 6: Random Forest model (CV probability prediction)

This step fits a random forest classifier and evaluates out-of-sample probability prediction using the same 5-fold stratified cross-validation and metrics as the logit baseline.

In [51]:
# Random Forest + CV evaluation

# Start timer
start_time = time.time()

# Random Forest classifier
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

# Pipeline: preprocessing + model
rf_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", rf_model),
])

# Cross-validated predicted probabilities
rf_proba = cross_val_predict(
    rf_pipe,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Runtime
runtime_rf = time.time() - start_time

# Performance metrics
rf_metrics = pd.DataFrame([{
    "model": "Random Forest",
    "roc_auc": roc_auc_score(y, rf_proba),
    "avg_precision": average_precision_score(y, rf_proba),
    "log_loss": log_loss(y, rf_proba),
    "brier": brier_score_loss(y, rf_proba),
    "runtime_sec": runtime_rf
}])

rf_metrics

,model,roc_auc,avg_precision,log_loss,brier,runtime_sec
0,Random Forest,0.827598,0.460313,0.274657,0.078075,3.541387


##### Step 7: LASSO Logit (L1) model (CV probability prediction)

This step fits a sparsity-inducing logit model (L1 regularization) as a third benchmark and evaluates probability prediction using the same cross-validation setup and metrics.

In [52]:
# LASSO Logit (L1)/ LASSO

# Start timer
start_time = time.time()

# Logistic regression with L1 regularization (sparse model)
logit_l1 = LogisticRegression(
    penalty="l1",
    solver="saga",
    max_iter=5000,
    n_jobs=-1
)

# Pipeline: preprocessing + model
logit_l1_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", logit_l1),
])

# Cross-validated predicted probabilities
logit_l1_proba = cross_val_predict(
    logit_l1_pipe,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Runtime
runtime_l1 = time.time() - start_time

# Performance metrics
logit_l1_metrics = pd.DataFrame([{
    "model": "Logit (L1 / LASSO)",
    "roc_auc": roc_auc_score(y, logit_l1_proba),
    "avg_precision": average_precision_score(y, logit_l1_proba),
    "log_loss": log_loss(y, logit_l1_proba),
    "brier": brier_score_loss(y, logit_l1_proba),
    "runtime_sec": runtime_l1
}])

logit_l1_metrics

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,model,roc_auc,avg_precision,log_loss,brier,runtime_sec
0,Logit (L1 / LASSO),0.752957,0.268564,0.432967,0.120592,742.482724


##### Step 8: Model comparison (horserace)

This step combines cross-validated performance metrics across the three candidate models to support model selection for the classification task.

In [53]:
# Step 8: Combine CV results (horserace table)
horserace = pd.concat([logit_l2_metrics, rf_metrics, logit_l1_metrics], ignore_index=True)

# Sort by ROC-AUC (primary), then runtime (secondary)
horserace = horserace.sort_values(["roc_auc", "runtime_sec"], ascending=[False, True]).reset_index(drop=True)

# Save table
overwrite_csv(
    horserace,
    os.path.join(output_tables, "task1_part1_horserace_cv_metrics.csv")
)

horserace

,model,roc_auc,avg_precision,log_loss,brier,runtime_sec
0,Random Forest,0.827598,0.460313,0.274657,0.078075,3.541387
1,Logit (L1 / LASSO),0.752957,0.268564,0.432967,0.120592,742.482724
2,Logit (L2),0.752954,0.268562,0.432966,0.120592,50.682915


##### Final Model Selection

The three models differ substantially in both predictive performance and computational cost.  
The random forest achieves the highest ROC–AUC and average precision, indicating superior ability to rank firms by fast-growth probability and to identify fast growers in an imbalanced setting. It also performs well in terms of log loss and Brier score, suggesting better-calibrated probabilities.

Both logit models (L2 and L1) perform similarly to each other but significantly worse than the random forest across all predictive metrics. While the L1-regularized logit offers automatic feature selection, it comes at a very high computational cost without improving predictive performance relative to the simpler L2 logit.

Overall, the random forest dominates the parametric alternatives by delivering stronger out-of-sample performance at a fraction of the runtime, making it the preferred model for probability prediction in this task.